# YOLOv8 + EasyOCR для распознавания ценников

Этот ноутбук обучает модель YOLOv8 на размеченных данных ценников и использует EasyOCR для распознавания текста.

## 1. Установка зависимостей

In [1]:
from rich.jupyter import display
!pip install -U ultralytics easyocr opencv-python pandas PyYAML

  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached ninja-1.13.0-py3-none-win_amd64.whl.metadata (5.1 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached tifffile-2026.3.3-py3-none-any.whl.metadata (31 kB)
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.2 MB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 3.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.9 MB 5.6 MB/s eta 0:00:01
   ------------------ --------------------- 1.3/2.9 MB 3.5 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.9 MB 3.5 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.9 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 3.3 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.3.13 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Импорты

In [2]:
import pathlib
import shutil
import csv
import cv2
import math
import random
import numpy as np
import pandas as pd
import easyocr
import matplotlib.pyplot as plt
from ultralytics import YOLO
import yaml
import logging
from typing import Dict, List, Tuple, Optional
import tempfile
import os

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

## 3. Подготовка данных для YOLOv8

In [5]:
def prepare_yolo_dataset():
    """Подготавливает датасет из YOLO darknet формата в ultralytics YOLOv8 формат"""
    
    data_root = pathlib.Path("data")
    obj_dir = data_root / "obj"
    yolo_data_dir = data_root / "yolo_dataset"
    yolo_data_dir.mkdir(exist_ok=True)
    
    (yolo_data_dir / "images" / "train").mkdir(parents=True, exist_ok=True)
    (yolo_data_dir / "images" / "val").mkdir(parents=True, exist_ok=True)
    (yolo_data_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)
    (yolo_data_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)
    
    train_list = set()
    val_list = set()
    
    with open(data_root / "train.txt") as f:
        for line in f:
            line = line.strip()
            if line:
                fname = pathlib.Path(line).name
                train_list.add(fname)
    
    with open(data_root / "test.txt") as f:
        for line in f:
            line = line.strip()
            if line:
                fname = pathlib.Path(line).name
                val_list.add(fname)
    
    for jpg_file in sorted(obj_dir.glob("*.jpg")):
        fname = jpg_file.name
        base_name = fname.replace(".jpg", "")
        txt_file = obj_dir / f"{base_name}.txt"
        
        if txt_file.exists():
            split = "train" if fname in train_list else ("val" if fname in val_list else "train")
            dst_img = yolo_data_dir / "images" / split / fname
            if not dst_img.exists():
                shutil.copy2(jpg_file, dst_img)
            dst_txt = yolo_data_dir / "labels" / split / f"{base_name}.txt"
            if not dst_txt.exists():
                shutil.copy2(txt_file, dst_txt)
    
    train_count = len(list((yolo_data_dir / "images" / "train").glob("*.jpg")))
    val_count = len(list((yolo_data_dir / "images" / "val").glob("*.jpg")))
    
    logger.info(f"Данные подготовлены в {yolo_data_dir}")
    logger.info(f"Train images: {train_count}, Val images: {val_count}")
    
    return yolo_data_dir

yolo_dataset_path = prepare_yolo_dataset()

[INFO] Данные подготовлены в data\yolo_dataset
[INFO] Train images: 60, Val images: 20


## 4. Создание YAML конфигурации датасета

In [7]:
def create_dataset_yaml(dataset_path: pathlib.Path) -> pathlib.Path:
    """Создаёт data.yaml для YOLOv8"""
    
    names_file = pathlib.Path("data/obj.names")
    with open(names_file) as f:
        classes = [line.strip() for line in f if line.strip()]
    
    yaml_content = {
        'path': str(dataset_path.absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(classes),
        'names': {i: name for i, name in enumerate(classes)}
    }
    
    yaml_path = dataset_path / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f)
    
    logger.info(f"Создан data.yaml: {yaml_path}")
    logger.info(f"Классы: {classes}")
    
    return yaml_path

yaml_path = create_dataset_yaml(yolo_dataset_path)

[INFO] Создан data.yaml: data\yolo_dataset\data.yaml
[INFO] Классы: ['description', 'barcode', 'price11', 'price22', 'price21', 'price12']


## 5. Инициализация YOLOv8 модели

In [8]:
model = YOLO('yolov8n.pt')  # Автоматически скачается при первом запуске
logger.info(f"Модель YOLOv8 загружена")

[INFO] Модель YOLOv8 загружена


## 6. Функция для вырезания ценников из видео

In [9]:
class PriceTagExtractor:
    """Вырезает ценники из видео по таймкодам и координатам"""
    
    def __init__(self, video_path: str, csv_path: str):
        self.video_path = pathlib.Path(video_path)
        self.csv_path = pathlib.Path(csv_path)
        self.video_data = self._load_csv()
        
    def _load_csv(self) -> pd.DataFrame:
        """Загружает CSV с координатами ценников"""
        try:
            df = pd.read_csv(self.csv_path, encoding='utf-8-sig')
            logger.info(f"[OK] Загружено {len(df)} записей из CSV")
            return df
        except Exception as e:
            logger.error(f"Ошибка при загрузке CSV: {e}")
            return pd.DataFrame()
    
    def extract_tags(self, output_dir: str = None) -> pathlib.Path:
        """Вырезает ценники по координатам из видео"""
        if len(self.video_data) == 0:
            logger.error("CSV данные не загружены")
            return None
        
        with tempfile.TemporaryDirectory() as td:
            temp_video = pathlib.Path(td) / 'video.mp4'
            shutil.copyfile(self.video_path, temp_video)
            cap = cv2.VideoCapture(str(temp_video))
            if not cap.isOpened():
                logger.error(f"Не удалось открыть видео: {temp_video}")
                return None
            
            fps = cap.get(cv2.CAP_PROP_FPS)
            logger.info(f"[INFO] Загружено видео: {self.video_path.name}  (FPS: {fps})")
            
            if output_dir is None:
                output_dir = self.video_path.parent / "extracted_price_tags"
            else:
                output_dir = pathlib.Path(output_dir)
            
            output_dir.mkdir(parents=True, exist_ok=True)
            
            grouped = self.video_data.groupby('frame_timestamp')
            extracted_count = 0
            metadata = []
            
            for frame_num, group in grouped:
                frame_num = int(frame_num)
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                ret, frame = cap.read()
                
                if not ret:
                    logger.warning(f"[WARN] Не удалось прочитать кадр {frame_num} из {self.video_path.name}")
                    continue
                
                for idx, row in group.iterrows():
                    try:
                        x_min = int(float(str(row['x_min']).replace(',', '.')))
                        y_min = int(float(str(row['y_min']).replace(',', '.')))
                        x_max = int(float(str(row['x_max']).replace(',', '.')))
                        y_max = int(float(str(row['y_max']).replace(',', '.')))
                        
                        x_min, x_max = min(x_min, x_max), max(x_min, x_max)
                        y_min, y_max = min(y_min, y_max), max(y_min, y_max)
                        
                        crop = frame[y_min:y_max, x_min:x_max]
                        
                        if crop.size > 0:
                            sku = str(row.get('id_sku', 'unknown')).replace('/', '_')
                            filename = f"{frame_num:06d}_sku_{sku}.jpg"
                            filepath = output_dir / filename
                            cv2.imwrite(str(filepath), crop)
                            extracted_count += 1
                            
                            metadata.append({
                                'filename': filename,
                                'frame': frame_num,
                                'sku': sku,
                                'product': row.get('product_name', ''),
                                'price': row.get('price_card', ''),
                                'x_min': x_min, 'y_min': y_min, 'x_max': x_max, 'y_max': y_max
                            })
                    except Exception as e:
                        logger.debug(f"Пропуск записи: {e}")
                        continue
            
            cap.release()
            logger.info(f"[DONE] Все ценники сохранены в папку: {output_dir}")
            
            if metadata:
                metadata_df = pd.DataFrame(metadata)
                metadata_path = output_dir / "extracted_tags_metadata.csv"
                metadata_df.to_csv(metadata_path, index=False, encoding='utf-8-sig')
                logger.info(f"[INFO] Создан файл с метаданными: {metadata_path}")
            
            return output_dir

logger.info("PriceTagExtractor инициализирован")

[INFO] PriceTagExtractor инициализирован


## 7. Инициализация EasyOCR

In [10]:
ocr_reader = easyocr.Reader(['ru', 'en'], gpu=False)
logger.info("EasyOCR инициализирован для [ru, en]")

[WARNING] Using CPU. Note: This module is much faster with a GPU.
[WARNING] Downloading detection model, please wait. This may take several minutes depending upon your network connection.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

[INFO] Download complete
[WARNING] Downloading recognition model, please wait. This may take several minutes depending upon your network connection.


Progress: |██████████████████████████████████████████████████| 100.1% Complete

[INFO] Download complete.
[INFO] EasyOCR инициализирован для [ru, en]


## 8. Функции для распознавания текста в ценниках

In [11]:
def recognize_price_text(image_path: str, reader=None) -> Dict[str, str]:
    """Распознаёт текст в ценнике с помощью EasyOCR"""
    if reader is None:
        reader = ocr_reader
    
    try:
        img = cv2.imread(image_path)
        if img is None:
            logger.warning(f"Не удалось загрузить изображение: {image_path}")
            return {}
        
        results = reader.readtext(img)
        texts = []
        for detection in results:
            text = detection[1]
            confidence = detection[2]
            if confidence > 0.3:
                texts.append(text)
        
        return {
            'raw_text': ' '.join(texts),
            'detections': len(results),
            'confidence_avg': np.mean([d[2] for d in results]) if results else 0
        }
    except Exception as e:
        logger.error(f"Ошибка при распознавании: {e}")
        return {}

def batch_recognize_price_tags(tags_dir: str, reader=None) -> pd.DataFrame:
    """Распознаёт текст во всех ценниках в директории"""
    if reader is None:
        reader = ocr_reader
    
    tags_path = pathlib.Path(tags_dir)
    results = []
    
    for img_path in sorted(tags_path.glob('*.jpg')):
        ocr_data = recognize_price_text(str(img_path), reader)
        results.append({
            'filename': img_path.name,
            'text': ocr_data.get('raw_text', ''),
            'confidence': ocr_data.get('confidence_avg', 0)
        })
    
    return pd.DataFrame(results)

logger.info("Функции OCR определены")

[INFO] Функции OCR определены


## 9. Конфигурация обучения YOLOv8

In [12]:
TRAINING_CONFIG = {
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'patience': 20,
    'device': 0,
    'optimizer': 'SGD',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'save': True,
    'save_period': -1,
    'verbose': True,
    'project': 'runs/detect',
    'name': 'yolo_price_tags',
    'exist_ok': False
}

logger.info("Конфигурация обучения подготовлена")

[INFO] Конфигурация обучения подготовлена


## 10. Загрузка обученных весов

In [20]:
def resolve_project_root() -> pathlib.Path:
    """Ищет корень проекта, чтобы корректно работать из Jupyter/Colab/IDE."""
    start = pathlib.Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "notebook" / "data").exists() and (candidate / "outputs").exists():
            return candidate
    return start


PROJECT_ROOT = resolve_project_root()
MANUAL_WEIGHTS_PATH = PROJECT_ROOT / "notebook" / "best.pt"


def find_weights_path() -> Optional[pathlib.Path]:
    """Ищет обученные веса в типичных местах; при необходимости можно добавить свой путь."""
    candidates = [
        MANUAL_WEIGHTS_PATH,
        PROJECT_ROOT / "models" / "best.pt",
        PROJECT_ROOT / "models" / "last.pt",
        PROJECT_ROOT / "weights" / "best.pt",
        PROJECT_ROOT / "weights" / "last.pt",
        PROJECT_ROOT / "best.pt",
        PROJECT_ROOT / "last.pt",
        PROJECT_ROOT / "runs" / "detect" / "yolo_price_tags" / "weights" / "best.pt",
        PROJECT_ROOT / "runs" / "detect" / "yolo_price_tags" / "weights" / "last.pt",
    ]
    for path in candidates:
        if path is not None and path.exists():
            return path
    logger.warning(
        "Не найдены веса модели. Положите best.pt/last.pt в models/ или обновите список candidates."
    )
    return None


WEIGHTS_PATH = find_weights_path()
trained_model = YOLO(str(WEIGHTS_PATH)) if WEIGHTS_PATH is not None else None
if trained_model is not None:
    logger.info(f"Загружены веса модели: {WEIGHTS_PATH}")

[INFO] Загружены веса модели: D:\Lenta-Tech-Life-Hack-2026\notebook\best.pt


## 11. Проверка качества на validation set

In [21]:
if trained_model is not None:
    val_metrics = trained_model.val(
        data=str(yaml_path),
        imgsz=640,
        batch=8,
        split="val",
        plots=True,
        verbose=True,
    )
    logger.info("Проверка на val завершена")
else:
    val_metrics = None
    logger.warning("Проверка на val пропущена: веса модели не найдены.")

Ultralytics 8.4.51  Python-3.11.9 torch-2.12.0+cpu CPU (13th Gen Intel Core i5-13500H)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 77.216.3 MB/s, size: 31.0 KB)
val: Scanning D:\Lenta-Tech-Life-Hack-2026\notebook\data\yolo_dataset\labels\val... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 1.1Kit/s 0.0s
val: New cache created: D:\Lenta-Tech-Life-Hack-2026\notebook\data\yolo_dataset\labels\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.5it/s 2.0s1.5s
                   all         20        120      0.946      0.987      0.995      0.864
           description         20         20      0.975          1      0.995      0.898
               barcode         20         20      0.979          1      0.995      0.977
               price11         20         20          1      0.919      0.995      0.861
              

[INFO] Проверка на val завершена


## 12. Визуализация предсказаний на `val`

In [27]:
def _sample_image_paths(folder: pathlib.Path, max_images: int = 6):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    paths = [p for p in sorted(folder.glob("*")) if p.suffix.lower() in exts]
    if len(paths) <= max_images:
        return paths
    return random.sample(paths, max_images)


def visualize_predictions_from_folder(
    model: YOLO,
    folder: str,
    max_images: int = 6,
    conf: float = 0.25,
    imgsz: int = 640,
    save_dir: str = None,
    title: str = "",
    rotate_ccw90: bool = False,
):
    folder_path = pathlib.Path(folder)
    if not folder_path.exists():
        logger.warning(f"Папка не найдена: {folder_path}")
        return []

    image_paths = _sample_image_paths(folder_path, max_images=max_images)
    if not image_paths:
        logger.warning(f"В папке нет изображений: {folder_path}")
        return []

    save_path = None
    if save_dir is not None:
        save_path = pathlib.Path(save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

    n = len(image_paths)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
    axes = np.array(axes).reshape(-1)

    saved_files = []
    for ax, img_path in zip(axes, image_paths):
        img = cv2.imread(str(img_path))
        if rotate_ccw90 and img is not None:
            img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
        results = model.predict(source=img, conf=conf, imgsz=imgsz, verbose=False)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated_rgb)
        ax.set_title(img_path.name, fontsize=10)
        ax.axis("off")

        if save_path is not None:
            out_file = save_path / img_path.name
            cv2.imwrite(str(out_file), annotated)
            saved_files.append(out_file)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title or folder_path.name, fontsize=14)
    plt.tight_layout()
    plt.show()

    return saved_files


VAL_IMAGES_DIR = PROJECT_ROOT / "notebook" / "data" / "yolo_dataset" / "images" / "val"
VAL_VIS_DIR = PROJECT_ROOT / "runs" / "visualizations" / "val_predictions"

if trained_model is not None:
    (visualize_predictions_from_folder(
        trained_model,
        folder=str(VAL_IMAGES_DIR),
        max_images=6,
        conf=0.25,
        imgsz=640,
        save_dir=str(VAL_VIS_DIR),
        title="Предсказания на validation set",
    ))
else:
    logger.warning("Визуализация на val пропущена: веса модели не найдены.")

<Figure size 1800x1200 with 6 Axes>

## 13. Визуализация на `outputs/extracted_price_tags_25_12_20`

In [29]:
EXTRACTED_TAGS_DIR = PROJECT_ROOT / "outputs" / "extracted_price_tags_25_12_20"
EXTRACTED_VIS_DIR = PROJECT_ROOT / "runs" / "visualizations" / "extracted_price_tags_25_12_20"

if trained_model is not None:
    _ = visualize_predictions_from_folder(
        trained_model,
        folder=str(EXTRACTED_TAGS_DIR),
        max_images=12,
        conf=0.25,
        imgsz=640,
        save_dir=str(EXTRACTED_VIS_DIR),
        title="Предсказания на extracted_price_tags_25_12_20",
        rotate_ccw90=True,
    )
else:
    logger.warning("Визуализация на extracted_price_tags_25_12_20 пропущена: веса модели не найдены.")

<Figure size 1800x2400 with 12 Axes>

## 14. ГОТОВО К ОБУЧЕНИЮ

Для начала обучения выполните следующую ячейку

In [ ]:
logger.info(f"START: Обучение YOLOv8 на датасете {yaml_path}")
logger.info(f"Dataset path: {yolo_dataset_path}")
logger.info(f"Training config: {TRAINING_CONFIG}")

# Раскомментируйте следующую строку для запуска обучения:
# results = model.train(data=str(yaml_path), **TRAINING_CONFIG)